In [2]:
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn

In [18]:
class NN(nn.Module):
    def __init__(self):
        super(NN, self).__init__()
        self.net = torch.nn.Sequential(
            nn.Linear(2, 20),
            nn.Tanh(),
            nn.Linear(20, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 20),
            nn.Tanh(),
            nn.Linear(20, 30),
            nn.Tanh(),
            nn.Linear(30, 1)
        )
    
    def forward(self, x):
        out = self.net(x)
        return out

In [29]:
class Net:
    def __init__(self):
        device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        
        self.h = 0.1
        self.k = 0.1

        x = torch.arange(-1, 1 + self.h, self.h)
        t = torch.arange(-1, 1 + self.k, self.k)

        self.X = torch.cartesian_prod(x, t) 

        bc1 = torch.cartesian_prod(torch.tensor([-1.0]), t)
        bc2 = torch.cartesian_prod(torch.tensor([1.0]), t)

        ic = torch.cartesian_prod(x, torch.tensor([-1.0]))

        self.X_train = torch.cat([bc1, bc2, ic], dim=0)

        y_bc1 = torch.zeros(len(bc1))
        y_bc2 = torch.zeros(len(bc2))
        y_ic = -torch.sin(math.pi * ic[:, 0])

        self.y_train = torch.cat([y_bc1, y_bc2, y_ic]).unsqueeze(1)

        self.X = self.X.to(device).float()
        self.X_train = self.X_train.to(device).float()
        self.y_train = self.y_train.to(device).float()
        
        self.X.requires_grad = True

        self.model = NN().to(device)

        self.optimizer_1 = torch.optim.Adam(self.model.parameters())
        self.optimizer_2 = torch.optim.LBFGS(self.model.parameters(), lr=1.0, max_iter=50000, max_eval=50000, history_size=50, tolerance_grad=1e-7, tolerance_change=1.0 * np.finfo(float).eps, line_search_fn="strong_wolfe")

        self.criterion = torch.nn.MSELoss()
        self.iter = 1 
    
    def loss_func(self):
        self.optimizer_1.zero_grad()
        self.optimizer_2.zero_grad()

        # 1. Boundary & Initial Condition Loss
        y_pred = self.model(self.X_train)
        loss_data = self.criterion(y_pred, self.y_train)

        # 2. PDE Residual Loss (Evaluate the model AT the collocation points self.X)
        u = self.model(self.X) 

        # First derivative
        du_dX = torch.autograd.grad(u, self.X, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
        du_dt = du_dX[:, 1]
        du_dx = du_dX[:, 0]

        # Second derivative
        du_dXX = torch.autograd.grad(du_dX, self.X, grad_outputs=torch.ones_like(du_dX), create_graph=True, retain_graph=True)[0]
        du_dxx = du_dXX[:, 0]

        pde_target = torch.zeros_like(du_dt)
        loss_pde = self.criterion(du_dt + u.squeeze() * du_dx - (0.01 / math.pi) * du_dxx, pde_target)
        
        # Total Loss
        loss = loss_pde + loss_data

        if self.iter % 100 == 0:
            print(f"Iteration {self.iter} - Loss: {loss.item():.6f}")
        self.iter = self.iter + 1

        return loss
    
    def train(self):
        self.model.train()

        for i in range(5000):
            self.optimizer_1.zero_grad()
            loss = self.loss_func()
            loss.backward()
            self.optimizer_1.step()

            if (i + 1) % 100 == 0:
                print(f"Adam Iteration {i+1} - Loss: {loss.item():.6e}")

        print("Starting LBFGS...")

        def closure():
            self.optimizer_2.zero_grad()
            loss = self.loss_func()
            loss.backward()

            return loss

        self.optimizer_2.step(closure)

        print("Finished LBFGS")
    def eval(self):
        self.model.eval()

In [30]:
net = Net()
net.train()

Iteration 100 - Loss: 0.100450
Adam Iteration 100 - Loss: 1.004499e-01
Iteration 200 - Loss: 0.064227
Adam Iteration 200 - Loss: 6.422737e-02
Iteration 300 - Loss: 0.056449
Adam Iteration 300 - Loss: 5.644938e-02
Iteration 400 - Loss: 0.049520
Adam Iteration 400 - Loss: 4.951990e-02
Iteration 500 - Loss: 0.032451
Adam Iteration 500 - Loss: 3.245070e-02
Iteration 600 - Loss: 0.020863
Adam Iteration 600 - Loss: 2.086317e-02
Iteration 700 - Loss: 0.015553
Adam Iteration 700 - Loss: 1.555254e-02
Iteration 800 - Loss: 0.013919
Adam Iteration 800 - Loss: 1.391865e-02
Iteration 900 - Loss: 0.015484
Adam Iteration 900 - Loss: 1.548372e-02
Iteration 1000 - Loss: 0.009030
Adam Iteration 1000 - Loss: 9.029967e-03
Iteration 1100 - Loss: 0.008248
Adam Iteration 1100 - Loss: 8.247524e-03
Iteration 1200 - Loss: 0.007306
Adam Iteration 1200 - Loss: 7.305684e-03
Iteration 1300 - Loss: 0.006606
Adam Iteration 1300 - Loss: 6.605562e-03
Iteration 1400 - Loss: 0.006301
Adam Iteration 1400 - Loss: 6.300757e

In [31]:
net.model.eval()

NN(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=20, bias=True)
    (1): Tanh()
    (2): Linear(in_features=20, out_features=30, bias=True)
    (3): Tanh()
    (4): Linear(in_features=30, out_features=30, bias=True)
    (5): Tanh()
    (6): Linear(in_features=30, out_features=20, bias=True)
    (7): Tanh()
    (8): Linear(in_features=20, out_features=30, bias=True)
    (9): Tanh()
    (10): Linear(in_features=30, out_features=1, bias=True)
  )
)

In [32]:
h = 0.01
k = 0.01

x = torch.arange(-1, 1, h)
t = torch.arange(0, 1, k)

X = torch.stack(torch.meshgrid(x,t)).reshape(2, -1).T
X = X.to(net.X.device)

In [33]:
model = net.model
model.eval()

with torch.no_grad():
    y_pred = model(X)
    y_pred = y_pred.reshape(len(x), len(t)).cpu().numpy()

In [34]:
sns.set_style("white")
plt.figure(figsize=(5, 3), dpi=3000)
sns.heatmap(y_pred, cmap='jet')

<Axes: >